Firstly,
1. Download nltk libraries
2. download spacy libraries

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

### News Article Analsis:
A Complete Pipeline project which would help user get insights on any news article.

In [ ]:
import numpy as np
import pandas as pd

#### 1. BBC News
This dataset contains news article from bbc news. Each article is also categorized in a single category based on its text.

#### 2. News Aggregation with summary from Narayan, Shashi, Shay B. Cohen, and Mirella Lapata. (2018).
This dataset contains news article from various news sources. Each article also have a human generated summary along with unique ID. Original dataset has 580014 article but that is way too large for our project. <br>
So, we would take a fraction of complete dataset.

In [ ]:
# Importing dataset
data = pd.read_csv("../Dataset/bbc-text.csv")
print("Shape: ",data.shape)
print("Five random sample points:")
data.sample(5)

## Main dataset for project
df = pd.read_csv("../Dataset/data.csv", nrows=10000, index_col=0)
df

In [ ]:
# # Importing dataset
# df = pd.read_excel("../Dataset/MN-DS-news-classification.xlsx")
# # Identify columns to drop
# unnamed_cols = [col for col in df.columns if 'Unnamed' in col]

# df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
# print("Shape: ",df.shape)
# print("Three random sample points:")
# df.sample(3)

In [ ]:
# # Converting date into valid readable format
# df['date'] = pd.to_numeric(df['date'], errors='coerce')
# df['date'] = pd.to_datetime(df['date'], unit='D', origin='1899-12-30')

### Now, we will analyse our dataset

First, Lets have a High level overview using Automatic EDA.

In [ ]:
# import dtale

# rep = dtale.show(df)
# rep

#### Step-1: High Level Overview

In [ ]:
# 1) Basic info and structure:
print("List of all columns: ",df.columns)
print("Column wise info: ",df.info())

In [ ]:
# 2) Datatype for each columns -> object
print("Datatype of each column: ",df.dtypes)

In [ ]:
# 3) Duplicates -> Remove them all
print("Number of duplicate rows in main dataset: ",df[df.duplicated()].shape[0])
print("Percentage of duplicate rows in main dataset: ",df[df.duplicated()].shape[0]/df.shape[0]*100)
dup_index = df.index[df.duplicated(keep='first')]
# df_clean = df.drop(dup_index)

## Similarily for bbc dataset
print("Number of duplicate rows in BBC dataset: ",data[data.duplicated()].shape[0])
print("Percentage of duplicate rows in BBC dataset: ",data[data.duplicated()].shape[0]/data.shape[0]*100)

In [ ]:
# 4) Missing values
missing_count = df.isnull().sum()
print("Missing Count:\n",missing_count)

missing_dataset = df[df.isnull().any(axis=1)]
print("\nNumber of rows with missing values: ",missing_dataset.shape[0])
print("\nPercentage of missing values: \n",missing_count/df.shape[0]*100)

In [ ]:
## Similarily for bbc dataset
missing_count = data.isnull().sum()
missing_dataset = data[data.isnull().any(axis=1)]
print("Number of rows with missing values: ",missing_dataset.shape[0])

Now, we will clean our dataset based off problem from high-level overview. <br>
-> Missing values + Duplicates + Textual problems + Type conversion

1. There are some article with no ID. As we have no need for any ID, we can just ignore these missing values.
2. As their are few duplicates, we can just remove them from our dataset. (dup < 5% of total dataset)
3. There are only three 3 defined source, we can convert this into category.
4. Rows with missing content are useless. Remove them from dataset.

In [ ]:
# looking into dataset source
print(df['Dataset'].value_counts())

# Since their are only three source, we can convert this into category
df['Dataset'] = df['Dataset'].astype('category')

# Removing duplicate
shape_before = df.shape
df = df.drop_duplicates(subset=['ID','Content'], keep='first')
print("Number of rows after removing duplicates: ",-df.shape[0] + shape_before[0])
shape_before = data.shape
data = data.drop_duplicates()
print("Number of rows after removing duplicates: ",-data.shape[0] + shape_before[0])

## Removing missing values
shape_before = df.shape
df = df.dropna(subset=['Content'])
print("Number of rows after removing missing values: ",-df.shape[0] + shape_before[0])

In [ ]:
## Removing unicodes:
import unicodedata, re
import html
from bs4 import BeautifulSoup

def clean_text(s: str) -> str:
    """
    Perform safe, canonical text cleaning for NLP tasks.
    Preserves linguistic structure.
    """
    if pd.isna(s) or not isinstance(s, str):
        return ''
    
    # 1. Fix broken encoding and HTML  entities
    s = html.unescape(s)    
    # 2. normalize unicode (NFKC helps)
    s = unicodedata.normalize('NFKC', s)
    # 3. Remove HTML tags (robust)
    s = BeautifulSoup(s, 'lxml').get_text(separator=" ")
    # 4. remove ZERO WIDTH and BOM chars
    s = re.sub(r'[\u200B-\u200D\uFEFF]', '', s)
    # 5. Normalize whitespace (spaces, tabs)
    s = re.sub(r"[ \t]+", " ", s)
    # 6. Remove repeated newlines
    s = re.sub(r"\n\s*\n+", "\n", s)
    # 7. Strip leading and trailing whitespace
    s = s.strip()
    
    return s


In [ ]:
# Create a cleaned column for diagnostics
df['Content'] = df['Content'].apply(clean_text)
df['Summary'] = df['Summary'].apply(clean_text)
data['text'] = data['text'].apply(clean_text)

In [ ]:
## Now, lets save these cleaned datasets
df.to_parquet('../Dataset/Clean/Clean_dataset.parquet')
data.to_parquet('../Dataset/Clean/Clean_bbc.parquet')

At this point we would look at other data cleaning problems like
* Spelling correction
* Outliers
* Data Transformation
* Imbalance for supervised task

But, 
- Our dataset is already checked for spelling mistake.
- There is no target label for Imbalance
- There are no outliers due to no numerical features yet
- There are no hardcore direct transformations required for complete raw text.

Now, We will process our textual data and create new features.

##### 🧱 A. Length & Size Features (FOUNDATIONAL)
These are always useful.
- char_count
- char_count_no_spaces
- word_count
- unique_word_count
- sentence_count

📌 Used in:
1. EDA
2. Quality checks
3. Readability
4. Feature dashboards

##### 🧠 B. Vocabulary & Lexical Richness
Measures how diverse the language is.
- lexical_diversity = unique_words / total_words
- hapax_legomena_ratio (words appearing once)
- top_k_word_ratio (dominance of common words)

📌 Useful for:
1. Article style analysis
2. Fake/low-quality detection
3. Comparing news outlets

##### 🚦 C. Stopword & Function Word Features
Stopwords are signal, not noise, for analysis.
- stopword_count
- stopword_ratio
- content_word_count
- content_word_ratio

📌 Useful for:
1. Readability
2. Writing style
3. Compression difficulty (summarization)

##### 🧾 D. Punctuation & Formatting Features [Not useful for our tasks]
Reflect writing structure.
- comma_count
- period_count
- question_mark_count
- exclamation_count
- quote_count
- punctuation_density

📌 Useful for:
1. Narrative vs factual tone
2. Opinion vs reporting

##### 🧠 E. POS (Part-of-Speech) Statistics
Very powerful, often ignored.
- noun_count
- verb_count
- adj_count
- adv_count
- pronoun_count
- pos_distribution

📌 Useful for:
1. Topic style
2. Abstraction level
3. Keyword extraction quality

##### 🧍 F. Named Entity Statistics
Blind, no labels required.
- person_count
- org_count
- gpe_count
- event_count
- unique_entity_count

📌 Useful for:
1. Newsworthiness
2. Topic richness
3. Similarity explanations

##### 📖 G. Readability & Complexity [Advance]
Classic but still relevant.
- flesch_reading_ease == Rate text reading ease [0-100]
- flesch_kincaid_grade == approximate the US grade level required to understand the text
- gunning_fog == Indicate years of formal education needed to understand the text

📌 Useful for:
1. User-facing insights
2. Comparing summaries vs originals

##### 🔁 H. Repetition & Redundancy [Not much useful]
- repeated_sentence_ratio
- repeated_word_ratio

📌 Useful for:
1. Detecting low-quality content
2. Summarization difficulty

In [ ]:
import spacy
import textstat
from collections import Counter

nlp = spacy.load('en_core_web_sm')
nlp_large = spacy.load('en_core_web_lg')

df = pd.read_parquet('../Dataset/Clean/Clean_dataset.parquet')
data = pd.read_parquet('../Dataset/Clean/Clean_bbc.parquet')

In [ ]:
## Completely spacy pipeline
def Extract_text_features(text: str) -> dict:
    doc = nlp(text)

    tokens = [t for t in doc if not t.is_space]
    words = [t for t in tokens if t.is_alpha]

    word_texts = [t.text.lower() for t in words]
    unique_words = set(word_texts)

    stopwords = [t for t in words if t.is_stop]
    content_words = [t for t in words if not t.is_stop]

    pos_counts = Counter(t.pos_ for t in words)
    ent_counts = Counter(ent.label_ for ent in doc.ents)

    features = {
        # Length
        "char_count": len(text),
        "char_count_no_spaces": len(text.replace(" ", "")),
        "word_count": len(words),
        "unique_word_count": len(unique_words),
        "sentence_count": len(list(doc.sents)),

        # Lexical
        "lexical_diversity": len(unique_words) / max(len(words), 1),
        "avg_word_length": sum(len(w) for w in word_texts) / max(len(words), 1),
        "avg_sentence_length": len(words) / max(len(list(doc.sents)), 1),

        # Stopwords
        "stopword_count": len(stopwords),
        "stopword_ratio": len(stopwords) / max(len(words), 1),
        "content_word_count": len(content_words),

        # POS
        "noun_count": pos_counts.get("NOUN", 0),
        "verb_count": pos_counts.get("VERB", 0),
        "adj_count": pos_counts.get("ADJ", 0),
        "adv_count": pos_counts.get("ADV", 0),
        "pronoun_count": pos_counts.get("PRON", 0),

        # Entities
        "person_count": ent_counts.get("PERSON", 0),
        "org_count": ent_counts.get("ORG", 0),
        "gpe_count": ent_counts.get("GPE", 0),
        "event_count": ent_counts.get("EVENT", 0),
        "unique_entity_count": len(set(ent.text for ent in doc.ents)),

        # Readability
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "gunning_fog": textstat.gunning_fog(text),
    }

    return features


Spacy is an easy to use library but it does not perform all these feature accurately. <br>
So, we will use other libraries to extract features.

In [ ]:
# NLP libs
import nltk

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from flair.models import SequenceTagger
from flair.data import Sentence

> Some features are not code due to correlation b/w independent variable aka Multicollinearity

In [ ]:
## 1. Text Length & Structure
def length_features(text:str) -> dict:
    sentences = sent_tokenize(text)
    words = [w for w in word_tokenize(text) if w.isalpha()]
    
    return {
        "char_count": len(text),
        "char_count_no_spaces": len(text.replace(" ", "")),
        "sentence_count": len(sentences),
    }

In [ ]:
## 2. Vocabulary & Lexical Richness [NLTK]
nltk.download('punkt')

def lexical_features(text:str) -> dict:
    tokens = [t.lower() for t in word_tokenize(text) if t.isalpha()]
    counts = Counter(tokens)
    
    rare_words = [w for w,c in counts.items() if c==1]
    
    return {
        "word_count": len(tokens),
        "unique_word_count": len(counts),
        "lexical_diversity": len(counts) / max(len(tokens), 1),
        # ratio of single occuring word/total words
        "hapax_ratio": len(rare_words) / max(len(tokens), 1)
    }

In [ ]:
## 3. Stopwords and Content-Word Features [NLTK]
nltk.download('stopwords')

stopword_list = set(stopwords.words('english'))

def stopword_features(text:str) -> dict:
    tokens = [t for t in word_tokenize(text) if t.isalnum()]
    stopwords_used = [t for t in tokens if t in stopword_list]
    
    return {
        "stopword_count": len(stopwords_used),
        "content_word_count": len(tokens) - len(stopwords_used),
        "stopword_ratio": len(stopwords_used) / max(1, len(tokens)),
    }

In [ ]:
## 4. POS Statistics [SpaCy]
def pos_features(text:str) -> dict:
    doc = nlp(text)
    pos_counts = Counter(tok.pos_ for tok in doc if tok.is_alpha)
    
    return {
        "noun_count":pos_counts.get("NOUN", 0),
        "verb_count":pos_counts.get("VERB", 0),
        "adj_count":pos_counts.get("ADJ", 0),
        "adv_count":pos_counts.get("ADV", 0),
        "pronoun_count":pos_counts.get("PRON", 0),
    }

In [ ]:
## 5. NER features [Spacy]
def ner_features(text:str) -> dict:
    doc = nlp(text)
    ent_counts = Counter(ent.label_ for ent in doc.ents)
    
    return {
        "person_count": ent_counts.get("PERSON", 0),
        "org_count": ent_counts.get("ORG", 0),
        "gpe_count": ent_counts.get("GPE", 0),
        "event_count": ent_counts.get("EVENT", 0),
        "unique_entity_count": len(set(ent.text for ent in doc.ents))
    }

tagger = SequenceTagger.load("flair/ner-english")

## 5. NER strong [Flair]
def _ner_features(text:str) -> dict:
    sentence = Sentence(text)
    tagger.predict(sentence)

    entities = [ent.get_label("ner").value for ent in sentence.get_spans("ner")]
    ent_counts = Counter(entities)

    return {
        "person_count": ent_counts.get("PER", 0),
        "org_count": ent_counts.get("ORG", 0),
        "location_count": ent_counts.get("LOC", 0),
        "misc_entity_count": ent_counts.get("MISC", 0),
        "total_entity_count": len(entities)
    }

In [ ]:
## 6. Readability Features [Textstat]
def readability_features(text:str) -> dict:
    return {
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),
        "gunning_fog": textstat.gunning_fog(text)
    }

In [ ]:
# Applying all on data
def extract_text_features(text:str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {}

    features = {}

    # 1. Length & structure
    features.update(length_features(text))
    # 2. Lexical richness
    features.update(lexical_features(text))
    # 3. Stopwords
    features.update(stopword_features(text))
    # 4. POS statistics (spaCy)
    features.update(pos_features(text))
    # 5. NER (choose ONE – Flair preferred for performance but spacy for speed)
    # features.update(_ner_features(text))   # stronger NER
    features.update(ner_features(text))  # spaCy alternative
    # 6. Readability
    features.update(readability_features(text))

    return features

In [ ]:
from tqdm import tqdm

def add_features_to_dataframe(DF:pd.DataFrame, text_col:str):
    """
    Apply NLP feature extractio to each row of any dataframe
    """
    if text_col not in DF.columns:
        raise ValueError(f"Column '{text_col}' not found in dataframe")
    
    texts = DF[text_col].fillna("").astype(str)
    features = texts.apply(extract_text_features)
    
    # Merge
    features_df = pd.DataFrame(features.tolist(), index=DF.index)
    df_enriched = pd.concat([DF, features_df], axis=1)
    
    return df_enriched

In [ ]:
group = data.groupby("text").count().sort_values("category", ascending=False)
group.head(2)

In [ ]:
## Getting features from data
# data_enrich = add_features_to_dataframe(data, 'text')
# data_enrich.sample(3)

In [ ]:
# ## Getting features from data
# data_enriched = add_features_to_dataframe(data, 'text')
# data_enriched.sample(3)

In [ ]:
## Getting features from df
# df_enriched = add_features_to_dataframe(df, 'Content')
# df_enriched.sample(3)

In [ ]:
## Getting features for summary column
# df_summary_enrich = add_features_to_dataframe(df, 'Summary')
# df_summary_enrich.sample(3)

In [ ]:
## Saving the dataset
# df_enriched.to_parquet('../Dataset/Clean/dataset_with_features.parquet')
# data_enriched.to_parquet('../Dataset/Clean/BBC_with_features.parquet')
# data_enrich.to_parquet('../Dataset/Clean/BBC_with_features_combined.parquet')
# df_summary_enrich.to_parquet('../Dataset/Clean/dataset_summary_features.parquet')